# SF Corpus Word2Vec by Era

Trains separate Word2Vec models for three historically-anchored slices of the corpus to track how environmental contamination is framed (accidental vs. human-caused) across the emergence of climate/environmental discourse in this period. Eras are cut around two real events rather than split evenly:

- `era_a` — **1945–1961**, pre-*Silent Spring*
- `era_b` — **1962–1971**, *Silent Spring* (1962) to just before the Clean Water Act
- `era_c` — **1972–1980**, Clean Water Act (1972) onward — grouped with the Earth Day/EPA-founding (1970) cluster of "the environmental turn"

These specific cutoffs (1962 / 1972, not 1962 / 1970) were chosen after checking the actual year distribution of the corpus: 1962/1972 gives nearly balanced eras (980 / 913 / 988 volumes), while 1962/1970 leaves the middle era noticeably thin (714) and the last era top-heavy (1187). 1972 is still historically defensible as the back edge of the same early-70s environmental-legislation cluster as 1970.

Adapted from `Temple_word2vec.ipynb`. Paths are left blank below — fill in `METADATA_CSV` and `TEXT_DIR` when running in the HathiTrust data capsule.

**Note:** if you're in a secure/offline capsule, run `nltk.download('punkt')` and `nltk.download('punkt_tab')` *before* entering secure mode (no internet access once inside), or copy the NLTK data directory in manually.

In [ ]:
#import libraries

from nltk.tokenize import sent_tokenize
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
import glob
import re
import random
import itertools
from pathlib import Path
import os
import gensim
from gensim.models.phrases import Phrases, Phraser
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from tqdm import tqdm
import multiprocessing
from collections import Counter

## Config

Two ways to point this at a corpus:
1. **One metadata CSV + one text directory** (default below): set `METADATA_CSV` (needs an id column and a year column) and `TEXT_DIR` (one `.txt` per volume, filename stem == id). The notebook sorts each volume into one of three fixed historical eras by year (cutoffs in `YEAR_CUTOFFS`), not an equal-count split.
2. **Already-split directories**: if you already have three separate folders, skip `assign_eras_by_year` and just set `era_assignments[label]["ids"] = None` with `TEXT_DIR` pointed at each folder in turn in the load-docs cell.


In [ ]:
# ---- CONFIG: fill in before running ----
# METADATA_CSV local source: Temple Project 2026/New_datasets_August2026/metadata_august2026.csv
# (exported from metadata_august2026.xlsx, confirmed to match the fully-audited 'combined' sheet in
# combining_lauretemple_novels_1945-1980.xlsx exactly -- 2881 unique htids, post 18-duplicate fix.
# Does NOT yet include the 2 confirmed genuine need_to_add titles (Foundation Trilogy, Empire Star)
# or a fix for the 21 laure_remaining_duplicates groups -- neither has been actioned yet.)
METADATA_CSV = "/home/dcuser/metadata_august2026.csv"   # path to a csv with an id column + a year column, one row per volume
TEXT_DIR = "/data/sf_corpus"        # directory containing one .txt per volume, filename stem == id
ID_COL = "htid"
YEAR_COL = "year"

ERA_LABELS = ["era_a", "era_b", "era_c"]  # earliest -> latest
# Fixed historical cutoffs, not equal-count buckets -- see top markdown cell for why these two years:
# era_a: year < YEAR_CUTOFFS[0]           (pre-Silent Spring)
# era_b: YEAR_CUTOFFS[0] <= year < YEAR_CUTOFFS[1]   (Silent Spring -> pre-Clean Water Act)
# era_c: year >= YEAR_CUTOFFS[1]          (Clean Water Act onward)
YEAR_CUTOFFS = [1962, 1972]

W2V_PARAMS = dict(
    min_count=2,
    vector_size=300,
    sg=1,  # skip-gram: tends to do better than CBOW on rarer words / smaller corpora
    # gensim's vocabulary build has a global lock, so throughput plateaus well before the
    # capsule's full core count (50 on dc6) -- capping avoids paying thread-contention
    # overhead for no speed benefit
    workers=min(os.cpu_count() or 4, 12),
)

# train each era multiple times with different seeds to check that neighbor-list
# findings are stable and not just noise from one training run (note: workers > 1
# means this isn't perfectly deterministic even with a fixed seed, but it still
# gives a useful stability signal)
SEEDS = [1, 2, 3]

## Split corpus into historically-anchored eras (1962 / 1972 cutoffs, not equal-count buckets)

In [ ]:
def assign_eras_by_year(df, id_col=ID_COL, year_col=YEAR_COL, cutoffs=None, labels=None):
    """Assign each row to a fixed historical era by year, not an equal-count split.

    cutoffs=[1962, 1972] with labels=[era_a, era_b, era_c] means:
      era_a: year <  1962
      era_b: 1962 <= year < 1972
      era_c: year >= 1972
    """
    labels = labels or ERA_LABELS
    cutoffs = cutoffs or YEAR_CUTOFFS
    assert len(labels) == len(cutoffs) + 1, "need exactly one more label than cutoff"

    bounds = [-float("inf")] + list(cutoffs) + [float("inf")]
    era_map = {}
    for label, lo, hi in zip(labels, bounds[:-1], bounds[1:]):
        bucket = df[(df[year_col] >= lo) & (df[year_col] < hi)]
        era_map[label] = {
            "ids": set(bucket[id_col].astype(str)),
            "year_min": int(bucket[year_col].min()) if len(bucket) else None,
            "year_max": int(bucket[year_col].max()) if len(bucket) else None,
            "n_docs": len(bucket),
        }
    return era_map


if METADATA_CSV:
    meta = pd.read_csv(METADATA_CSV)
    era_assignments = assign_eras_by_year(meta)
    for label, info in era_assignments.items():
        print(f"{label}: {info['n_docs']} docs, years {info['year_min']}-{info['year_max']}")
else:
    era_assignments = {label: {"ids": None, "year_min": None, "year_max": None, "n_docs": None} for label in ERA_LABELS}
    print("METADATA_CSV not set — fill in CONFIG before running.")

## Load + clean + preprocess

In [ ]:
tokenizer = TreebankWordTokenizer()

# alphabetic tokens only (allows an internal apostrophe, e.g. "don't"), length > 1 —
# strips OCR scanno garbage, stray punctuation, running-header/page-number noise
# that's common in HathiTrust-scanned text
TOKEN_RE = re.compile(r"^[a-z]+(?:'[a-z]+)?$")


def clean_tokens(tokens):
    return [t for t in tokens if TOKEN_RE.match(t) and len(t) > 1]


def load_docs(base_dir, ids=None):
    """Read HTRC capsule text: base_dir contains one subfolder PER HTID (e.g.
    mdp.39015002644014/), and each htid folder contains one .txt file PER PAGE
    (00000065.txt, 00000217.txt, ...), not one .txt per volume. Concatenates all
    pages within an htid folder, in filename order, into a single document per
    volume. Loose top-level files like volume-rights.txt / volumes_not_available.txt
    are HTRC workset metadata, not book text, and are skipped since they aren't
    inside an htid-named folder. If ids is given, only load htid folders whose
    name is in ids."""
    all_docs = []
    base_dir = Path(base_dir)
    for htid_dir in sorted(p for p in base_dir.iterdir() if p.is_dir() and not p.name.startswith(".")):
        htid = htid_dir.name
        if ids is not None and htid not in ids:
            continue
        pages = sorted(htid_dir.glob("*.txt"))
        if not pages:
            continue
        text = "\n".join(p.read_text(encoding="utf-8", errors="replace") for p in pages)
        all_docs.append(text)
    return all_docs


def make_sentences(list_text):
    """Lowercase, sentence-split, word-tokenize, then strip OCR noise from each sentence."""
    all_sentences = []
    for txt in tqdm(list_text, desc="Preprocessing"):
        lower_txt = txt.lower()
        sentences = sent_tokenize(lower_txt)
        for sent in sentences:
            cleaned = clean_tokens(tokenizer.tokenize(sent))
            if cleaned:
                all_sentences.append(cleaned)
    return all_sentences

In [ ]:
sentences_by_era = {}
for label in ERA_LABELS:
    if not TEXT_DIR:
        print(f"Skipping {label}: TEXT_DIR not set")
        continue
    ids = era_assignments[label]["ids"]
    docs = load_docs(TEXT_DIR, ids=ids)
    print(f"{label}: {len(docs)} documents loaded")
    sentences_by_era[label] = make_sentences(docs)

## Corpus balance: report + subsample to equal token counts

A model trained on more text gets tighter, more "reliable-looking" neighbor lists regardless of any real semantic change. Downsampling every era to the token count of the smallest era keeps the cross-era comparison fair.

In [ ]:
def token_count(sentences):
    return sum(len(s) for s in sentences)


def report_token_counts(sentences_by_era_dict):
    counts = {}
    for era, sents in sentences_by_era_dict.items():
        c = token_count(sents)
        counts[era] = c
        print(f"{era}: {c:,} tokens, {len(sents):,} sentences")
    return counts


def subsample_to_match(sentences_by_era_dict, seed=1):
    """Randomly downsample sentences (no replacement) in every era to match the smallest era's token count."""
    rng = random.Random(seed)
    counts = {era: token_count(s) for era, s in sentences_by_era_dict.items()}
    target = min(counts.values())
    balanced = {}
    for era, sents in sentences_by_era_dict.items():
        shuffled = sents[:]
        rng.shuffle(shuffled)
        kept, total = [], 0
        for sent in shuffled:
            if total >= target:
                break
            kept.append(sent)
            total += len(sent)
        balanced[era] = kept
    return balanced


report_token_counts(sentences_by_era)
balanced_sentences = subsample_to_match(sentences_by_era) if sentences_by_era else {}
report_token_counts(balanced_sentences)

## Bigram phrase detection

Fit on the combined corpus (all eras together) so phrase tokens like `acid_rain`, `greenhouse_effect`, `population_bomb`, `nuclear_war` are recognized consistently across eras — fitting `Phrases` separately per era would let the same two words get merged in one era and stay separate in another, which would bias the comparison.

In [ ]:
def build_phraser(sentences_by_era_dict, min_count=5, threshold=10):
    combined = list(itertools.chain.from_iterable(sentences_by_era_dict.values()))
    phrases = Phrases(combined, min_count=min_count, threshold=threshold)
    return Phraser(phrases)


def apply_phraser(phraser, sentences):
    return [phraser[sent] for sent in sentences]


phrased_sentences = {}
if balanced_sentences:
    phraser = build_phraser(balanced_sentences)
    phrased_sentences = {era: apply_phraser(phraser, sents) for era, sents in balanced_sentences.items()}

## Keyword list

Grouped by the historical/critical hooks for the environment-in-New-Wave question, not just alphabetically.

In [ ]:
ENV_WORD_GROUPS = {
    "landscape_baseline": ["river", "creek", "stream", "water", "forest", "nature", "wilderness", "jungle",
                            "ocean", "landscape", "levee", "dam", "reservoir", "estuary", "wetland",
                            "marsh", "watershed"],
    "ecology_concept": ["ecology", "ecosystem", "environment", "biosphere", "habitat", "balance", "cycle"],
    "contamination": ["radiation", "radioactive", "fallout", "contamination", "waste", "smog", "fumes",
                       "chemical", "pesticide", "exhaust", "toxic", "polluted", "pollution"],
    "waste_infrastructure": ["sewer", "sewage", "drainage", "effluent", "runoff", "wastewater",
                              "cesspool", "sludge", "septic", "cistern", "culvert", "plumbing"],
    "population_scarcity": ["overpopulation", "population", "famine", "scarcity", "starvation", "resource", "drought"],
    "energy": ["oil", "fuel", "energy", "coal", "nuclear", "power"],
    "disaster_collapse": ["wasteland", "extinction", "collapse", "barren", "dying", "decay", "catastrophe"],
    "climate_weather": ["climate", "weather", "warming", "greenhouse", "atmosphere", "temperature",
                         "flood", "flooding", "storm", "hurricane", "ice", "glacier", "carbon", "ozone"],
    "space_earth_framing": ["earth", "homeworld", "colony", "frontier", "terraform", "alien"],
}

ENV_WORDS = [w for group in ENV_WORD_GROUPS.values() for w in group]

# word-pair contrasts: for each entry, compares sim(a, b1) vs sim(a, b2) per era —
# a cheap proxy for how a term's framing shifts (e.g. earth-as-home vs earth-as-resource)
WORD_PAIR_CONTRASTS = [
    ("earth", "home", "earth", "resource"),
    ("nature", "machine", "nature", "technology"),
    ("earth", "garden", "earth", "wasteland"),
]

## Frequency baseline (before touching embeddings)

A much more legible first pass than word2vec neighbor shifts, and a sanity check against them — if a keyword's raw frequency barely changes across eras, a neighbor-list shift for that word deserves more scrutiny before it becomes an argument.

In [ ]:
from matplotlib import pyplot as plt


def freq_per_million(sentences, words):
    counts = Counter(tok for sent in sentences for tok in sent)
    total = sum(counts.values()) or 1
    return {w: (counts[w] / total) * 1_000_000 for w in words}


def plot_frequency_trends(sentences_by_era_dict, words, era_order=None):
    era_order = era_order or list(sentences_by_era_dict.keys())
    data = {era: freq_per_million(sentences_by_era_dict[era], words) for era in era_order}
    df = pd.DataFrame(data, index=words).T
    df.plot(marker="o", figsize=(10, 6))
    plt.ylabel("Frequency per million tokens")
    plt.title("Keyword frequency across eras")
    plt.xticks(range(len(era_order)), era_order)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    return df


if phrased_sentences:
    freq_df = plot_frequency_trends(phrased_sentences, ENV_WORDS)

## Train Word2Vec (multiple seeds per era for stability)

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec


class EpochProgress(CallbackAny2Vec):
    """Updates a tqdm bar once per training epoch (gensim defaults to 5 epochs, since
    W2V_PARAMS doesn't override it)."""
    def __init__(self, total_epochs, desc):
        self.pbar = tqdm(total=total_epochs, desc=desc, leave=False)

    def on_epoch_end(self, model):
        self.pbar.update(1)

    def on_train_end(self, model):
        self.pbar.close()


models_by_era_seed = {}  # (era, seed) -> model
run_combos = [(era, seed) for era in phrased_sentences for seed in SEEDS]

for era, seed in tqdm(run_combos, desc="era/seed runs"):
    params = dict(W2V_PARAMS, seed=seed)
    epoch_cb = EpochProgress(total_epochs=params.get("epochs", 5), desc=f"{era} seed{seed} epochs")
    model = gensim.models.Word2Vec(phrased_sentences[era], callbacks=[epoch_cb], **params)
    model.save(f"sf_model_{era}_seed{seed}")
    models_by_era_seed[(era, seed)] = model

# primary model per era (first seed) used for the main analysis below
models = {era: models_by_era_seed[(era, SEEDS[0])] for era in phrased_sentences}

## Reload saved models

Run this cell on its own in a later session instead of retraining.

In [ ]:
from gensim.models import Word2Vec

models_by_era_seed = {}
for era in ERA_LABELS:
    for seed in SEEDS:
        path = f"sf_model_{era}_seed{seed}"
        if os.path.exists(path):
            models_by_era_seed[(era, seed)] = Word2Vec.load(path)

models = {era: models_by_era_seed[(era, SEEDS[0])] for era in ERA_LABELS if (era, SEEDS[0]) in models_by_era_seed}
list(models.keys())

## Cross-era neighbor + similarity comparisons

**Caveat:** these compare neighbor lists and in-model similarity scores, which is valid — but raw vectors from separately-trained models are *not* directly comparable (different random initialization). Don't compute cosine distance between a word's `era_a` vector and its `era_c` vector without aligning the spaces first (see the optional Procrustes section below).

In [ ]:
def most_similar_by_era(word, topn=5):
    for era, model in models.items():
        if word not in model.wv:
            print(f"{era}: '{word}' not in vocabulary")
            continue
        print(f"{era}: {model.wv.most_similar(word, topn=topn)}")


def similarity_by_era(word_a, word_b):
    rows = []
    for era, model in models.items():
        sim = model.wv.similarity(word_a, word_b) if word_a in model.wv and word_b in model.wv else None
        rows.append((era, sim))
    return pd.DataFrame(rows, columns=["era", f"sim({word_a}, {word_b})"])


def pair_contrast_by_era(word_a, word_b1, word_b2):
    """Compare sim(word_a, word_b1) vs sim(word_a, word_b2) across eras."""
    rows = []
    for era, model in models.items():
        sim1 = model.wv.similarity(word_a, word_b1) if word_a in model.wv and word_b1 in model.wv else None
        sim2 = model.wv.similarity(word_a, word_b2) if word_a in model.wv and word_b2 in model.wv else None
        rows.append((era, sim1, sim2, (sim1 - sim2) if sim1 is not None and sim2 is not None else None))
    return pd.DataFrame(rows, columns=["era", f"sim({word_a},{word_b1})", f"sim({word_a},{word_b2})", "diff"])

In [ ]:
for word in ENV_WORDS:
    print(f"--- {word} ---")
    most_similar_by_era(word)
    print()

In [ ]:
for word_a, word_b1, _, word_b2 in WORD_PAIR_CONTRASTS:
    display(pair_contrast_by_era(word_a, word_b1, word_b2))

## Neighbor stability check across seeds

Before building an argument on a neighbor-list finding (e.g. "toxic → radioactive"), check that it's stable across training runs and not an artifact of one particular initialization.

In [ ]:
def neighbor_stability(word, era, topn=10):
    """Average Jaccard overlap of top-N neighbor sets across seeds for one word/era."""
    neighbor_sets = []
    for seed in SEEDS:
        model = models_by_era_seed.get((era, seed))
        if model is None or word not in model.wv:
            continue
        neighbor_sets.append({w for w, _ in model.wv.most_similar(word, topn=topn)})
    if len(neighbor_sets) < 2:
        return None
    pairs = list(itertools.combinations(neighbor_sets, 2))
    jaccards = [len(a & b) / len(a | b) for a, b in pairs]
    return sum(jaccards) / len(jaccards)


# example: check stability for every env word, every era
stability_rows = []
for era in models:
    for word in ENV_WORDS:
        stability_rows.append((era, word, neighbor_stability(word, era)))
stability_df = pd.DataFrame(stability_rows, columns=["era", "word", "jaccard_stability"]).dropna()
stability_df.sort_values("jaccard_stability").head(10)  # least-stable words to treat cautiously

## KWIC / concordance grounding

For whatever neighbor shifts turn out strongest, pull actual passages containing the word — grounds the quantitative claim in a citable line.

In [ ]:
def build_concordance_index(sentences):
    flat = list(itertools.chain.from_iterable(sentences))
    return nltk.Text(flat)


concordance_by_era = {era: build_concordance_index(sents) for era, sents in phrased_sentences.items()}


def kwic(word, era, width=80, lines=10):
    concordance_by_era[era].concordance(word, width=width, lines=lines)


# example (uncomment once models/text are loaded):
# kwic("toxic", "era_a")

## Optional: aligning embeddings for direct cross-era vector comparison

Everything above compares neighbor lists and in-model similarity scores, which doesn't require alignment. If you want to make a claim like "the vector for *toxic* moved closer to *nature* between era_a and era_c," the two models' vector spaces need to be aligned first (orthogonal Procrustes, following Hamilton et al.'s diachronic embedding approach) — otherwise the comparison is meaningless since each model's coordinate system comes from an independent random initialization.

In [ ]:
from scipy.linalg import orthogonal_procrustes


def align_vectors(base_model, other_model):
    """Align other_model's space onto base_model's via orthogonal Procrustes over shared vocabulary."""
    shared_vocab = [w for w in base_model.wv.index_to_key if w in other_model.wv]
    base_matrix = np.array([base_model.wv[w] for w in shared_vocab])
    other_matrix = np.array([other_model.wv[w] for w in shared_vocab])
    R, _ = orthogonal_procrustes(other_matrix, base_matrix)
    return {w: other_model.wv[w] @ R for w in other_model.wv.index_to_key}


def semantic_shift(word, base_model, aligned_vectors):
    """Cosine distance a word's vector moved between base_model's era and the aligned era."""
    if word not in base_model.wv or word not in aligned_vectors:
        return None
    v1, v2 = base_model.wv[word], aligned_vectors[word]
    cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return 1 - cos_sim


# example (uncomment once models are trained):
# aligned_c_onto_a = align_vectors(models["era_a"], models["era_c"])
# semantic_shift("toxic", models["era_a"], aligned_c_onto_a)

## t-SNE plot, one era at a time

In [ ]:
from sklearn.manifold import TSNE


def plot_era(era, base_words=("toxic", "pollution", "river", "water"), topn=5):
    model = models[era]

    similar_by_base = {
        w: [i[0] for i in model.wv.most_similar(positive=w, topn=topn)]
        for w in base_words
    }
    all_words = np.hstack(list(similar_by_base.values()) + [list(base_words)])

    labels = list(all_words)
    tokens = model.wv[labels]

    tsne_model = TSNE(init="pca", learning_rate="auto", perplexity=15)
    new_values = tsne_model.fit_transform(tokens)

    x = [v[0] for v in new_values]
    y = [v[1] for v in new_values]

    colors = plt.cm.tab10.colors
    color_by_base = {w: colors[i % len(colors)] for i, w in enumerate(base_words)}

    plt.figure()
    for i, word in enumerate(labels):
        plt.annotate(word, xy=(x[i], y[i]), xytext=(5, 2), textcoords="offset points", ha="right", va="bottom")
        owning_base = next((w for w in base_words if word == w or word in similar_by_base[w]), None)
        plt.scatter(x[i], y[i], color=color_by_base.get(owning_base, "black"))
    plt.title(f"Word2Vec Embeddings — {era}", fontweight="bold")
    plt.show()

In [ ]:
# plot_era("era_a")
# plot_era("era_b")
# plot_era("era_c")